## Full Model

In [ ]:
import pandas as pd
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import ast # CSV에 저장된 문자열 리스트를 실제 리스트로 변환하기 위해 필요
from sklearn.model_selection import KFold 

In [ ]:
# --- 0. (필수) 경로 설정 ---
CSV_PATH = r'C:\BRAINEEWHA\processed_zmaps\final_multimodal_dataset.csv'
BASE_DIR = Path(r'C:\BRAINEEWHA\processed_zmaps')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# --- 1. Custom Dataset 정의 ---
def _safe_nifti_path(p_str: str) -> str:
    """
    '.nii' vs '.nii.gz' 및 파일명 변형(예: *_wm_zmap.nii) 자동 보정.
    """
    p = Path(str(p_str))
    if p.exists():
        return str(p)
    # .nii.gz -> .nii
    if p.suffixes == ['.nii', '.gz']:
        alt = p.with_suffix('')  # .gz 제거
        if alt.exists():
            return str(alt)
    # .nii -> .nii.gz
    if p.suffix == '.nii':
        alt = Path(str(p) + '.gz')
        if alt.exists():
            return str(alt)
    # 같은 폴더에서 '...zmap...' 후보 탐색 (예: *_wm_zmap.nii)
    stem = p.stem
    prefix = stem.split('_zmap')[0]
    hits = list(p.parent.glob(prefix + '*zmap*.nii*'))
    if hits:
        return str(hits[0])
    # 마지막으로 subject 힌트(숫자 6자리)로 넓게 탐색
    subj = re.findall(r'\d{6}', stem)
    if subj:
        hits = list(p.parent.glob(f"*{subj[0]}*zmap*.nii*"))
        if hits:
            return str(hits[0])
    raise FileNotFoundError(f"zmap 파일을 찾을 수 없음: {p_str}")

In [ ]:
# --- 2. Multimodal Fusion 모델 정의 ---
def parse_behavioral_features(feature_string: str) -> np.ndarray:
    """
    CSV에 저장된 문자열 벡터를 float32 numpy array로 파싱.
    - '[0.1, -0.2, ...]' 또는 '[ 0.1 -0.2 ... ]' 모두 처리
    """
    if feature_string is None or (isinstance(feature_string, float) and np.isnan(feature_string)):
        return np.array([], dtype=np.float32)
    s = str(feature_string).strip()
    # 1차: literal_eval 시도
    try:
        obj = ast.literal_eval(s)
        arr = np.array(obj, dtype=np.float32).ravel()
        return arr
    except Exception:
        pass
    # 2차: 정규식으로 숫자 추출
    nums = re.findall(r'[-+]?\d*\.\d+|[-+]?\d+', s)
    try:
        arr = np.array([float(x) for x in nums], dtype=np.float32).ravel()
        return arr
    except Exception:
        return np.array([], dtype=np.float32)

In [ ]:
# --- 3. CSV 로드 + 경로/파싱/길이 정리 → full_df, NUM_BEHAV_FEATURES ---
full_df = pd.read_csv(CSV_PATH)

# zmap 경로 보정(존재하는 것만 남김)
def fix_row_path(row):
    p = row.get('zmap_path', '')
    try:
        return _safe_nifti_path(p)
    except Exception:
        # subject 열이 있으면 보조 탐색
        sid = str(row.get('subject', ''))
        if sid:
            cand = list(BASE_DIR.glob(f"*{sid}*zmap*.nii*"))
            if cand:
                return str(cand[0])
        return ''

if 'zmap_path' not in full_df.columns:
    raise KeyError("CSV에 'zmap_path' 열이 없습니다.")
if 'wm_accuracy' not in full_df.columns:
    raise KeyError("CSV에 'wm_accuracy' 열이 없습니다.")
if 'behavioral_features' not in full_df.columns:
    raise KeyError("CSV에 'behavioral_features' 열이 없습니다.")

full_df['zmap_path'] = full_df.apply(fix_row_path, axis=1)
full_df = full_df[full_df['zmap_path'].map(lambda s: Path(s).exists())].reset_index(drop=True)

# 타깃 결측 제거
full_df = full_df.dropna(subset=['wm_accuracy']).reset_index(drop=True)

# 행동 벡터 파싱 및 길이 통일(모드값)
behav_arrays = full_df['behavioral_features'].map(parse_behavioral_features)
lengths = behav_arrays.map(len)
if lengths.eq(0).any():
    print("[경고] 빈 행동 벡터가 있어 제거합니다:", int((lengths==0).sum()))
mode_len = int(lengths.mode().iat[0]) if not lengths.mode().empty else 0
full_df = full_df[(lengths == mode_len) & (lengths > 0)].reset_index(drop=True)

# 전역 피처 차원 (K-Fold 코드에서 사용: NUM_BEHAV_FEATURES)
NUM_BEHAV_FEATURES = mode_len
print("유효 샘플 수:", len(full_df))
print("행동 feature 차원(NUM_BEHAV_FEATURES):", NUM_BEHAV_FEATURES)

In [ ]:
# 3-2) Dataset (K-Fold 코드와 인터페이스 동일: HCPDataset(df))
class HCPDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe.copy()
        # 미리 파싱해 캐시 (길이 검증은 전처리 단계에서 이미 완료)
        self._behavs = [parse_behavioral_features(s) for s in self.df['behavioral_features']]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1) Z-map
        zmap_path = _safe_nifti_path(row['zmap_path'])
        zmap = nib.load(zmap_path).get_fdata().astype(np.float32)   # (D,H,W)
        z_tensor = torch.from_numpy(zmap)[None, ...]                 # (C=1,D,H,W)

        # 2) 행동 벡터
        bvec = torch.from_numpy(self._behavs[idx].astype(np.float32))  # (F,)

        # 3) 타깃
        y = torch.tensor(row['wm_accuracy'], dtype=torch.float32)

        return z_tensor, bvec, y


# 3-3) 모델 (K-Fold에서 MultimodalFusionModel(num_behavioral_features=NUM_BEHAV_FEATURES)로 호출)
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_behavioral_features: int):
        super().__init__()
        # 3D-CNN branch
        self.cnn_branch = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(8, 16, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1), nn.ReLU(),
            # Flatten 전 출력 공간 크기 고정 → 32 * 11 * 13 * 11
            nn.AdaptiveAvgPool3d((11, 13, 11)),
            nn.Flatten(),
        )
        self.cnn_output_features = 32 * 11 * 13 * 11  # = 50336

        # MLP branch
        self.mlp_branch = nn.Sequential(
            nn.Linear(num_behavioral_features, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 64), nn.ReLU()
        )
        self.mlp_output_features = 64

        # Fusion head
        self.classifier_head = nn.Sequential(
            nn.Linear(self.cnn_output_features + self.mlp_output_features, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x_img, x_behav):
        img_features = self.cnn_branch(x_img)          # (N, 50336)
        behav_features = self.mlp_branch(x_behav)      # (N, 64)
        combined = torch.cat((img_features, behav_features), dim=1)
        out = self.classifier_head(combined)
        return out.squeeze(1)  # (N,1) -> (N,)


Using device: cpu
유효 샘플 수: 10
행동 feature 차원(NUM_BEHAV_FEATURES): 8


In [ ]:
# --- 4. K-Fold 교차 검증 루프 (N_SPLITS=5) ---
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# 각 Fold의 검증 R-squared를 저장할 리스트
fold_val_r2_scores = []
fold_val_losses = []

print(f"--- 🚀 {N_SPLITS}-Fold 교차 검증 시작 ---")

# full_df.index를 [train_idx], [val_idx]로 분할
for fold, (train_indices, val_indices) in enumerate(kf.split(full_df)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    # 4.1. Fold별 데이터 분할 (4명 / 1명)
    train_df = full_df.iloc[train_indices]
    val_df = full_df.iloc[val_indices]
    
    print(f"Train: {len(train_df)} samples, Validation: {len(val_df)} samples")

    # 4.2. Dataset 및 DataLoader 생성
    train_dataset = HCPDataset(train_df)
    val_dataset = HCPDataset(val_df)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True) # 4명뿐이므로 batch_size=4
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False) # 1명이므로 batch_size=1

    # 4.3. 모델, 손실 함수, 옵티마이저 초기화
    # ‼️ 중요: 매 Fold마다 새 모델로 "초기화"해야 합니다.
    model = MultimodalFusionModel(num_behavioral_features=NUM_BEHAV_FEATURES).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    NUM_EPOCHS = 50 # 각 Fold마다 30 Epochs씩 학습
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        
        for zmaps, behavs, targets in train_loader:
            zmaps, behavs, targets = zmaps.to(DEVICE), behavs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(zmaps, behavs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * zmaps.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)

        # (Epoch마다 Val R-squared를 찍으면 N=1이라 nan이 나옴)
        if (epoch + 1) % 10 == 0: # 10 Epoch마다 Train Loss만 출력
             print(f"  Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {epoch_loss:.4f}")

    # 4.4. Fold 학습 완료 후 최종 검증 (Validation)
    model.eval()
    final_val_loss = 0.0
    all_targets = []
    all_outputs = []
    
    with torch.no_grad():
        for zmaps, behavs, targets in val_loader: # (N=1, 1번만 실행됨)
            zmaps, behavs, targets = zmaps.to(DEVICE), behavs.to(DEVICE), targets.to(DEVICE)
            outputs = model(zmaps, behavs)
            loss = criterion(outputs, targets)
            final_val_loss += loss.item() * zmaps.size(0)
            
            all_targets.extend(targets.cpu().numpy())
            all_outputs.extend(outputs.cpu().numpy())
            
    final_val_loss = final_val_loss / len(val_loader.dataset)
    
    # ‼️ N=1이라 R-squared는 계산 불가 (nan), Loss만 기록
    # val_r2 = r2_score(all_targets, all_outputs) # (이 코드는 N=1이라 nan 반환)
    
    print(f"  Fold {fold+1} 완료. 최종 Val Loss: {final_val_loss:.4f}")
    fold_val_losses.append(final_val_loss)


print("\n--- ✅ 교차 검증 완료 ---")
print(f"각 Fold의 Val Loss: {fold_val_losses}")
print(f"평균 Val Loss (MSE): {np.mean(fold_val_losses):.4f}")
print(f"평균 Val RMSE: {np.sqrt(np.mean(fold_val_losses)):.4f}")

--- 🚀 5-Fold 교차 검증 시작 ---

--- Fold 1/5 ---
Train: 8 samples, Validation: 2 samples
  Epoch 10/50 - Train Loss: 4893.3767
  Epoch 20/50 - Train Loss: 721.4176
  Epoch 30/50 - Train Loss: 599.4822
  Epoch 40/50 - Train Loss: 296.4279
  Epoch 50/50 - Train Loss: 283.0988
  Fold 1 완료. 최종 Val Loss: 17.6561

--- Fold 2/5 ---
Train: 8 samples, Validation: 2 samples
  Epoch 10/50 - Train Loss: 4185.3042
  Epoch 20/50 - Train Loss: 1252.1702
  Epoch 30/50 - Train Loss: 178.1010
  Epoch 40/50 - Train Loss: 463.2221
  Epoch 50/50 - Train Loss: 267.5354
  Fold 2 완료. 최종 Val Loss: 31.4369

--- Fold 3/5 ---
Train: 8 samples, Validation: 2 samples
  Epoch 10/50 - Train Loss: 3500.6051
  Epoch 20/50 - Train Loss: 300.4911
  Epoch 30/50 - Train Loss: 381.2491
  Epoch 40/50 - Train Loss: 144.6381
  Epoch 50/50 - Train Loss: 271.6304
  Fold 3 완료. 최종 Val Loss: 152.4017

--- Fold 4/5 ---
Train: 8 samples, Validation: 2 samples
  Epoch 10/50 - Train Loss: 5555.3025
  Epoch 20/50 - Train Loss: 728.3419
  Epo

## Ablation Study

In [38]:
# 2.2. ‼️ (Ablation 모델 1) fMRI-Only 모델
class fMRI_Only_Model(nn.Module):
    def __init__(self, cnn_output_size=50336):
        super(fMRI_Only_Model, self).__init__()
        self.cnn_output_features = cnn_output_size
        
        # CNN 브랜치는 원본과 동일
        self.cnn_branch = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, stride=1, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(8, 16, kernel_size=3, stride=1, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Flatten(),
        )
        # ‼️ MLP 브랜치가 없음
        # ‼️ Classifier Head가 CNN 출력만 받음
        self.classifier_head = nn.Sequential(
            nn.Linear(self.cnn_output_features, 128), # 입력 크기가 다름
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 1)
        )
    def forward(self, x_img): # ‼️ 입력으로 x_img만 받음
        img_features = self.cnn_branch(x_img)
        output = self.classifier_head(img_features)
        return output.squeeze(1)

In [39]:
# 2.3. ‼️ (Ablation 모델 2) Behavioral-Only 모델
class Behavioral_Only_Model(nn.Module):
    def __init__(self, num_behavioral_features, mlp_output_size=64):
        super(Behavioral_Only_Model, self).__init__()
        self.mlp_output_features = mlp_output_size

        # ‼️ CNN 브랜치가 없음
        # MLP 브랜치는 원본과 동일
        self.mlp_branch = nn.Sequential(
            nn.Linear(num_behavioral_features, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, mlp_output_size), nn.ReLU(),
        )
        # ‼️ Classifier Head가 MLP 출력만 받음
        self.classifier_head = nn.Sequential(
            nn.Linear(self.mlp_output_features, 128), # 입력 크기가 다름
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 1)
        )
    def forward(self, x_behav): # ‼️ 입력으로 x_behav만 받음
        behav_features = self.mlp_branch(x_behav)
        output = self.classifier_head(behav_features)
        return output.squeeze(1)

In [40]:
# --- 3. 데이터 로드 ---
print("데이터 로드 중...")
try:
    full_df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"‼️ 오류: {CSV_PATH} 파일을 찾을 수 없습니다.")
    exit()

if len(full_df) < 5:
    print(f"‼️ 오류: K-Fold(N_SPLITS=5)를 위해 최소 5개의 샘플이 필요합니다.")
    exit()

데이터 로드 중...


In [41]:
# --- 4. K-Fold 교차 검증 루프 (N_SPLITS=5) ---
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# ‼️ 각 모델의 Fold별 최종 Val Loss를 저장할 리스트
fold_val_losses_full = []
fold_val_losses_fmri = []
fold_val_losses_behav = []

print(f"--- 🚀 {N_SPLITS}-Fold 교차 검증 시작 (Ablation Study) ---")

for fold, (train_indices, val_indices) in enumerate(kf.split(full_df)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    
    # 4.1. Fold별 데이터 분할 (4명 / 1명)
    train_df = full_df.iloc[train_indices]
    val_df = full_df.iloc[val_indices]
    
    train_dataset = HCPDataset(train_df)
    val_dataset = HCPDataset(val_df)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    # 4.2. ‼️ 3개 모델 및 옵티마이저 초기화
    model_full = MultimodalFusionModel(num_behavioral_features=NUM_BEHAV_FEATURES).to(DEVICE)
    model_fmri = fMRI_Only_Model().to(DEVICE)
    model_behav = Behavioral_Only_Model(num_behavioral_features=NUM_BEHAV_FEATURES).to(DEVICE)
    
    optimizer_full = optim.Adam(model_full.parameters(), lr=0.0001)
    optimizer_fmri = optim.Adam(model_fmri.parameters(), lr=0.0001)
    optimizer_behav = optim.Adam(model_behav.parameters(), lr=0.0001)
    
    criterion = nn.MSELoss()
    NUM_EPOCHS = 50

    for epoch in range(NUM_EPOCHS):
        # ‼️ 3개 모델 모두 학습 모드로
        model_full.train()
        model_fmri.train()
        model_behav.train()
        
        for zmaps, behavs, targets in train_loader:
            zmaps, behavs, targets = zmaps.to(DEVICE), behavs.to(DEVICE), targets.to(DEVICE)
            
            # --- Full Model 학습 ---
            optimizer_full.zero_grad()
            outputs_full = model_full(zmaps, behavs)
            loss_full = criterion(outputs_full, targets)
            loss_full.backward()
            optimizer_full.step()
            
            # --- fMRI-Only Model 학습 ---
            optimizer_fmri.zero_grad()
            outputs_fmri = model_fmri(zmaps) # zmaps만 입력
            loss_fmri = criterion(outputs_fmri, targets)
            loss_fmri.backward()
            optimizer_fmri.step()
            
            # --- Behavioral-Only Model 학습 ---
            optimizer_behav.zero_grad()
            outputs_behav = model_behav(behavs) # behavs만 입력
            loss_behav = criterion(outputs_behav, targets)
            loss_behav.backward()
            optimizer_behav.step()
        
        # (학습 중 로그는 생략, Epoch 30 정도는 금방 끝납니다)
        if (epoch + 1) % 10 == 0:
             print(f"  Epoch {epoch+1}/{NUM_EPOCHS} 진행 중...")

    # 4.3. ‼️ Fold 학습 완료 후 3개 모델 최종 검증 (Validation)
    model_full.eval()
    model_fmri.eval()
    model_behav.eval()
    
    val_loss_full = 0.0
    val_loss_fmri = 0.0
    val_loss_behav = 0.0
    
    with torch.no_grad():
        for zmaps, behavs, targets in val_loader: # (N=1, 1번만 실행됨)
            zmaps, behavs, targets = zmaps.to(DEVICE), behavs.to(DEVICE), targets.to(DEVICE)
            
            # Full
            outputs_full = model_full(zmaps, behavs)
            val_loss_full += criterion(outputs_full, targets).item() * zmaps.size(0)
            
            # fMRI-Only
            outputs_fmri = model_fmri(zmaps)
            val_loss_fmri += criterion(outputs_fmri, targets).item() * zmaps.size(0)
            
            # Behavioral-Only
            outputs_behav = model_behav(behavs)
            val_loss_behav += criterion(outputs_behav, targets).item() * zmaps.size(0)

    # 4.4. ‼️ 각 모델의 최종 Loss를 리스트에 추가
    fold_val_losses_full.append(val_loss_full / len(val_loader.dataset))
    fold_val_losses_fmri.append(val_loss_fmri / len(val_loader.dataset))
    fold_val_losses_behav.append(val_loss_behav / len(val_loader.dataset))
    
    print(f"  Fold {fold+1} 완료.")
    print(f"    - Full Model Val Loss: {val_loss_full/len(val_loader.dataset):.4f}")
    print(f"    - fMRI-Only Val Loss: {val_loss_fmri/len(val_loader.dataset):.4f}")
    print(f"    - Behav-Only Val Loss: {val_loss_behav/len(val_loader.dataset):.4f}")

--- 🚀 5-Fold 교차 검증 시작 (Ablation Study) ---

--- Fold 1/5 ---
  Epoch 10/50 진행 중...
  Epoch 20/50 진행 중...
  Epoch 30/50 진행 중...
  Epoch 40/50 진행 중...
  Epoch 50/50 진행 중...
  Fold 1 완료.
    - Full Model Val Loss: 83.3458
    - fMRI-Only Val Loss: 10.0218
    - Behav-Only Val Loss: 8250.7297

--- Fold 2/5 ---
  Epoch 10/50 진행 중...
  Epoch 20/50 진행 중...
  Epoch 30/50 진행 중...
  Epoch 40/50 진행 중...
  Epoch 50/50 진행 중...
  Fold 2 완료.
    - Full Model Val Loss: 34.1449
    - fMRI-Only Val Loss: 169.4941
    - Behav-Only Val Loss: 7892.7383

--- Fold 3/5 ---
  Epoch 10/50 진행 중...
  Epoch 20/50 진행 중...
  Epoch 30/50 진행 중...
  Epoch 40/50 진행 중...
  Epoch 50/50 진행 중...
  Fold 3 완료.
    - Full Model Val Loss: 30.5445
    - fMRI-Only Val Loss: 55.7152
    - Behav-Only Val Loss: 8601.1206

--- Fold 4/5 ---
  Epoch 10/50 진행 중...
  Epoch 20/50 진행 중...
  Epoch 30/50 진행 중...
  Epoch 40/50 진행 중...
  Epoch 50/50 진행 중...
  Fold 4 완료.
    - Full Model Val Loss: 71.0645
    - fMRI-Only Val Loss: 42.6317
    -

In [45]:
# --- 5. ‼️ 최종 결과 분석 ---
print("\n--- ✅ 최종 결과 분석 ---")

avg_loss_full = np.mean(fold_val_losses_full)
avg_loss_fmri = np.mean(fold_val_losses_fmri)
avg_loss_behav = np.mean(fold_val_losses_behav)

print("\n--- 1. 모델 예측 성능 평가 (5-Fold CV 평균) ---")
print(f"Full Model (fMRI + Behav) : 평균 Val Loss (MSE) = {avg_loss_full:.4f} (RMSE = {np.sqrt(avg_loss_full):.4f})")
print(f"fMRI-Only Model           : 평균 Val Loss (MSE) = {avg_loss_fmri:.4f} (RMSE = {np.sqrt(avg_loss_fmri):.4f})")
print(f"Behavioral-Only Model     : 평균 Val Loss (MSE) = {avg_loss_behav:.4f} (RMSE = {np.sqrt(avg_loss_behav):.4f})")

print("\n--- 2. 데이터 기여도 분석 ---")
if avg_loss_fmri < avg_loss_behav and avg_loss_fmri < avg_loss_full:
    print("➡️ 결론: fMRI-Only 모델의 성능이 가장 좋습니다.")
    print("   이는 fMRI 데이터가 예측에 가장 크게 기여하며, 행동 데이터는 오히려 방해가 되었을 수 있음을 시사합니다.")
elif avg_loss_behav < avg_loss_fmri and avg_loss_behav < avg_loss_full:
    print("➡️ 결론: Behavioral-Only 모델의 성능이 가장 좋습니다.")
    print("   이는 행동 데이터가 예측에 가장 크게 기여하며, fMRI 데이터는 오히려 방해가 되었을 수 있음을 시사합니다.")
elif avg_loss_full < avg_loss_fmri and avg_loss_full < avg_loss_behav:
    print("➡️ 결론: Full Model (Fusion)의 성능이 가장 좋습니다.")
    print("   이는 fMRI와 행동 데이터가 상호 보완적으로 작용하여, 두 데이터를 결합했을 때 가장 좋은 예측 성능을 보인다는 것을 의미합니다.")
else:
    print("➡️ 결론: 세 모델의 성능이 유사하거나 Full Model의 이점이 명확하지 않습니다.")
    print("   데이터가 더 필요하거나 모델 아키텍처(퓨전 방식)의 수정이 필요할 수 있습니다.")


--- ✅ 최종 결과 분석 ---

--- 1. 모델 예측 성능 평가 (5-Fold CV 평균) ---
Full Model (fMRI + Behav) : 평균 Val Loss (MSE) = 48.7841 (RMSE = 6.9846)
fMRI-Only Model           : 평균 Val Loss (MSE) = 64.8592 (RMSE = 8.0535)
Behavioral-Only Model     : 평균 Val Loss (MSE) = 8190.8234 (RMSE = 90.5032)

--- 2. 데이터 기여도 분석 ---
➡️ 결론: Full Model (Fusion)의 성능이 가장 좋습니다.
   이는 fMRI와 행동 데이터가 상호 보완적으로 작용하여, 두 데이터를 결합했을 때 가장 좋은 예측 성능을 보인다는 것을 의미합니다.
